# PC-FNO3D NATIONAL v4, candidat D2 (D + canal d'innovation) — graine 42 (Colab, A100)

Règle `configs/national/regle_d2_national.json` (sha256 `c7c98a20ab22…`), vérifiée par le carnet ; code au commit `5767232`. **Uniquement avec l'accord écrit d'Ines** (accord du 11/11 au soir pour les 3 entraînements, transmis par le coordinateur). Vérité d'entraînement v3 (`cache_v3_partie_*`).

Ce carnet : statistiques d'innovation (jours d'entraînement) et choix de H (validation, OI seule), puis entraînement (30 époques, recette de D, départ v2) et évaluation d'information (validation). **Aucune lecture du test.**

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "576723242e1c689afa0a3084beb5fb8fe6770d65"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

assert hashlib.sha256((CODE / "configs" / "national" / "regle_d2_national.json").read_bytes()).hexdigest() \
    == "c7c98a20ab2200326248c65bf0c1a6574c71f68a704ac97ee13b065d47621836", "regle D2 modifiee"
print("regle D2 verifiee")

## 2. Cache v3

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. Mémoire du GPU

In [ ]:
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/national_1km_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Statistiques, H, entraînement, information, fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import shlex
from google.colab import runtime
import traceback
GRAINE = 42
RUN = NAT / "runs" / f"national_1km_v4_D2_s{GRAINE}"
RUN.mkdir(parents=True, exist_ok=True)
DEGV4 = NAT / "runs" / "degradation_era5_v4.json"          # celle de D (regle v4), jours d'entrainement
STATS = NAT / "runs" / "statistiques_innovation_d2.json"   # L, sigma_o / sigma_b (jours d'entrainement)
REG = NAT / "runs" / "reglage_H_d2.json"                   # H (validation, OI seule, sans reseau)

def attendre(f, heures=4):
    """Graines 43 et 44 : les JSON communs sont ecrits par le carnet de la graine 42 (jamais deux fois)."""
    t0 = time.time()
    while not f.exists():
        if time.time() - t0 > heures * 3600:
            raise TimeoutError(f"{f} absent apres {heures} h : lancer d'abord le carnet de la graine 42")
        print(f"attente de {f.name} (carnet de la graine 42)...", flush=True)
        time.sleep(120)

try:
    if GRAINE == 42:
        if not DEGV4.exists():
            lancer("degradation_era5_v4", f"python -u scripts/degradation_era5.py --donnees {G} "
                                          f"--calendrier {NAT / 'calendrier.json'} --sortie {DEGV4}")
        if not STATS.exists():
            lancer("statistiques_innovation_d2", f"python -u scripts/statistiques_innovation_d2.py --donnees {G} "
                                                 f"--calendrier {NAT / 'calendrier.json'} --sortie {STATS}")
    attendre(DEGV4)
    attendre(STATS)
    st = json.loads(STATS.read_text(encoding="utf-8"))
    print({k: st[k] for k in ("L_km", "rapport_o_b", "sigma_b", "sigma_o", "L_en_butee", "distance_min_km", "paires")})
    INN = (f"--set innovation.actif=true --set innovation.L_km={st['L_km']:.3f} "
           f"--set innovation.rapport_o_b={st['rapport_o_b']:.6f} --set mask.aneo_hauteur_capteur_m=10")
    if GRAINE == 42 and not REG.exists():
        lancer("reglage_H_d2", f"python -u scripts/info_capteurs_d2.py reglage -c national_1km_colab {OPTS} {INN} "
                               f"--degradation {shlex.quote(str(DEGV4))} --sortie {shlex.quote(str(REG))} "
                               "--H 200,500,1000 --condition propre")
    attendre(REG)
    H = float(json.loads(REG.read_text(encoding="utf-8"))["H_retenu_m"])
    INN += f" --set innovation.H_m={H:g}"
    print("H retenu (validation) :", H, "m | innovation :", INN)
    DEG = RUN / "degradation_era5.json"
    if not DEG.exists():
        shutil.copy2(DEGV4, DEG)
    INNJ = RUN / "innovation_d2.json"
    if not INNJ.exists():
        INNJ.write_text(json.dumps({"regle_sha256": "c7c98a20ab2200326248c65bf0c1a6574c71f68a704ac97ee13b065d47621836", "L_km": st["L_km"], "rapport_o_b": st["rapport_o_b"],
                                    "H_m": H, "L_en_butee": st["L_en_butee"], "statistiques": str(STATS),
                                    "reglage_H": str(REG)}, indent=2), encoding="utf-8")
    EXTRA = ("--set loss.lambda_rough=1.0 "
             f"--set mask.aneo_degradation={shlex.quote(str(DEG))} --set mask.aneo_p_zone=0.5 "
             f"--set {shlex.quote('mask.aneo_capteurs=[20,200]')} {INN}")
    INIT = shlex.quote(str(choisir_poids(NAT / "runs" / "national_1km_v2")))
    print("depart :", INIT, "| run :", RUN)
    SEL = shlex.quote('train.select_criteres=["dso1_rmse","dso2_masse","dso3_rugosite"]')
    lancer(f"entrainer_v4_D2_s{GRAINE}",
           f"python -u -m pcfno3d -c national_1km_colab {OPTS} {EXTRA} --set {SEL} --set train.sync_last=false "
           f"--set train.init_from={INIT} --set train.lr=1e-4 --set train.seed={GRAINE} "
           f"--set paths.run_dir={shlex.quote(str(RUN))} --set paths.local_dir=/content/run_local_v4_D2_s{GRAINE} train")
    # AUCUNE lecture du test ici (regle_d2_national.json)
    print((RUN / "history.json").read_text()[-3000:])
    # information, sans effet sur le choix : D2, ERA5 seule et OI seule ; propre / degradee / + 50 capteurs
    if not (RUN / "info_capteurs_d2_validation.json").exists():
        lancer(f"info_capteurs_d2_s{GRAINE}",
               f"python -u scripts/info_capteurs_d2.py info -c national_1km_colab {OPTS} {INN} "
               f"--run {shlex.quote(str(RUN))} --degradation {shlex.quote(str(DEGV4))}")
    print((RUN / "info_capteurs_d2_validation.json").read_text()[:4000])
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()